In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import math

In [ ]:
sc.settings.verbosity = 3 
sc.logging.print_header()
sc.settings.set_figure_params(dpi=300, transparent = True, format = 'pdf', vector_friendly = True)

In [ ]:
figure = "Figure_4"

In [ ]:
sc.settings.figdir = './Figure_plots/'+figure

In [ ]:
umap_cmap = sns.blend_palette(['xkcd:light grey', 'xkcd:blueberry'], as_cmap = True)

# input files

In [ ]:
adata = sc.read_h5ad('h5ad/analysis_250528_f/Smed_L78-L47_20250523_Annotated.h5ad')

In [ ]:
adata

In [ ]:
# output from 01_G1_vs_G2_DGE.Rmd
res = pd.read_excel('h5ad/analysis_250528_f/DGE/2C_4C/outputs/G1_vs_G2_diffExpGenes_results_all.xlsx', index_col = 0)

In [ ]:
res['log_pval'] = -np.log10(res['pvalue'])

In [ ]:
res

In [ ]:
# lists of genes differentially expressed in G1 and G2
li_G1 = res[(res['padj'] < 0.05) & ((res['log2FoldChange'] < 0))].index.to_list()
li_G2 = res[(res['padj'] < 0.05) & ((res['log2FoldChange'] > 0))].index.to_list()

In [ ]:
sc.pl.umap(adata, color = li_G1, cmap = umap_cmap)

In [ ]:
sc.pl.umap(adata, color = li_G2, cmap = umap_cmap)

In [ ]:
G1_res_annot = res.loc[li_G1][['baseMean', 'log2FoldChange', 'lfcSE', 'stat', 'pvalue', 'padj']].merge(adata.raw.var.loc[li_G1][['longest_isoform', 'gene_type', 
       'gene_ddv6', 'Preferred_name', 'Description.x', 'PFAMs', 'Class']],   left_index=True,
    right_index=True)

In [ ]:
G2_res_annot = res.loc[li_G2][[
    'baseMean', 'log2FoldChange', 'lfcSE', 'stat', 'pvalue', 'padj'
]].merge(adata.raw.var.loc[li_G2][[
    'longest_isoform', 'gene_type', 'gene_ddv6', 'Preferred_name', 'Description.x', 'PFAMs', 'Class'
]],   left_index=True,
    right_index=True)

In [ ]:
with pd.ExcelWriter("../Files Supplementary/Supplementary File 4  G1G2 DGE/G1_vs_G2_diffExpGenes_results_all.xlsx", engine="openpyxl") as writer:
    G1_res_annot.to_excel(writer, sheet_name="G1")
    G2_res_annot.to_excel(writer, sheet_name="G2")

In [ ]:
# many of the differentialy expressed genes are expressed in progenitors
# intersect with genes enriched in neoblasts

In [ ]:
neo_enrich_df = pd.concat([
    pd.DataFrame(adata.uns['broad']['names'])['neoblasts'].rename('names'),
    pd.DataFrame(adata.uns['broad']['scores'])['neoblasts'].rename('scores'),
    pd.DataFrame(adata.uns['broad']['pvals'])['neoblasts'].rename('pvals'),
    pd.DataFrame(adata.uns['broad']['pvals_adj'])['neoblasts'].rename('pvals_adj'),
    pd.DataFrame(adata.uns['broad']['logfoldchanges'])['neoblasts'].rename('logfoldchanges'),
    
], axis = 1).set_index('names')

In [ ]:
neoblast_list = neo_enrich_df[(neo_enrich_df['logfoldchanges'] > 0) & (neo_enrich_df['pvals_adj'] < 0.05)].index.to_list()

In [ ]:
G1_li_f = [i for i in li_G1 if i in neoblast_list]
G2_li_f = [i for i in li_G2 if i in neoblast_list]

In [ ]:
len(G2_li_f)

In [ ]:
sc.pl.umap(adata, color = G1_li_f, cmap = umap_cmap)

In [ ]:
sc.pl.umap(adata, color = G2_li_f, cmap = umap_cmap)

In [ ]:
adata.raw.var.loc[G1_li_f][['Preferred_name', 'Description.x']]

In [ ]:
adata.raw.var.loc[G2_li_f][['Preferred_name', 'Description.x']]

In [ ]:
res.loc[G1_li_f + G2_li_f, "deg_neo"] = "DEG"
res["deg_neo"] = res["deg_neo"].fillna("non sig.")

In [ ]:
# create a new column for the colors of the volcano plot
res['volcano'] = res['deg']
res.loc[G1_li_f, 'volcano'] = "G1_neo"
res.loc[G2_li_f, 'volcano'] = "G2_neo"

In [ ]:
res["log2FoldChange"].max()

In [ ]:
res["log2FoldChange"].min()

In [ ]:
# Set figure size
plt.figure(figsize=(7, 8.5))

# Scatter plot
sns.scatterplot(
    data=res,
    x="log2FoldChange",
    y="log_pval",
    hue="volcano", 
    alpha=0.8, 
    edgecolor='grey',
    palette={'DEG': 'black', 'none': 'grey', 'G1_neo': 'darkred', 'G2_neo' : 'gold'},  
    s=70 , 
    legend=False
)

plt.axvline(x=0, color='black', linestyle='--', linewidth=1.5, zorder=1)

plt.grid(True)     
plt.gca().set_axisbelow(True)  
plt.xlim(-5.3, 5.3)

plt.xlabel("log2 fold change", fontsize=21)
plt.ylabel("-log10(pval)", fontsize=21)

plt.tight_layout()

plt.savefig( './Figure_plots/'+figure + "/volcano_plot.pdf", format="pdf", bbox_inches="tight")

plt.show()


In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(25, 15), dpi=300)
adata_from_raw = adata.raw.to_adata()

# (row, col): gene
positions = {
    (0, 0): "h1SMcG0000152", # G1 neo
    (0, 1): "h1SMcG0003975", # G1 neo
    (0, 2): "h1SMcG0008035", # G2 neo
    (0, 3): "h1SMcG0006230", # G2 neo
    (0, 4): "h1SMcG0013999", # G2 neo

    (1, 0): "h1SMcG0017068", # G1 neo
    (1, 1): "h1SMcG0023142", # G1 neo
    (1, 2): "h1SMcG0016303", # G2 neo
    (1, 3): "h1SMcG0018166", # G2 neo
    (1, 4): "h1SMcG0019482", # G2 neo

    (2, 0): "h1SMcG0002269", # G1
    (2, 1): "h1SMcG0005537", # G1
    (2, 2): "h1SMcG0009633", # G1
    (2, 3): "h1SMcG0007437", # G2 
    (2, 4): "h1SMcG0011383", # G2
}

for ax in axes.flatten():
    ax.axis("off")

for (row, col), gene in positions.items():
    sc.pl.umap(
        adata_from_raw,
        color=gene,
        cmap=umap_cmap,
        s=8,
        ax=axes[row, col],
        show=False,
        title=gene
    )
    axes[row, col].set_axis_off()
    axes[row, col].title.set_fontsize(22)
    axes[row, col].title.set_fontweight("bold")

plt.tight_layout()
plt.savefig(f'./Figure_plots/{figure}/{figure}_umaps_2.pdf')
plt.show()


In [ ]:
# umaps for the other genes enriched in neoblasts
G2_li_neo = [i for i in G2_li_f if i not in ["h1SMcG0008035", "h1SMcG0006230", "h1SMcG0013999", "h1SMcG0016303","h1SMcG0018166", "h1SMcG0019482"]]
G1_li_neo = [i for i in G1_li_f if i not in ["h1SMcG0000152", "h1SMcG0003975", "h1SMcG0017068", "h1SMcG0023142"]]

In [ ]:
n = len(G1_li_neo)
ncols = 5 
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows), dpi=300)

axes = axes.flatten()
adata_from_raw = adata.raw.to_adata()

# Turn off all axes first
for ax in axes:
    ax.axis("off")

# Plot
for i, gene in enumerate(G1_li_neo):
    sc.pl.umap(
        adata_from_raw,
        color=gene,
        cmap=umap_cmap,
        s=10,
        ax=axes[i],
        show=False,
        title=gene
    )
    axes[i].set_axis_off()
    axes[i].title.set_fontsize(20)
    axes[i].title.set_fontweight("bold")

plt.tight_layout()
plt.savefig('./Figure_plots/Figure_sup_G1G2DGE/G1_neo_umaps.pdf')
plt.show()


In [ ]:
n = len(G2_li_neo)
ncols = 5 
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows), dpi=300)

axes = axes.flatten()
adata_from_raw = adata.raw.to_adata()

# Turn off all axes first
for ax in axes:
    ax.axis("off")

# Plot
for i, gene in enumerate(G2_li_neo):
    sc.pl.umap(
        adata_from_raw,
        color=gene,
        cmap=umap_cmap,
        s=10,
        ax=axes[i],
        show=False,
        title=gene
    )
    axes[i].set_axis_off()
    axes[i].title.set_fontsize(20)
    axes[i].title.set_fontweight("bold")

plt.tight_layout()
plt.savefig('./Figure_plots/Figure_sup_G1G2DGE/G2_neo_umaps.pdf')
plt.show()


In [ ]:
G2_rest = [i for i in li_G2 if i not in G2_li_f and i not in ["h1SMcG0007437", "h1SMcG0011383"]]
G1_rest = [i for i in li_G1 if i not in G1_li_f and i not in ["h1SMcG0002269", "h1SMcG0005537", "h1SMcG0009633"]]


In [ ]:
n = len(G1_rest + G2_rest)
ncols = 5 
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows), dpi=300)

axes = axes.flatten()
adata_from_raw = adata.raw.to_adata()

# Turn off all axes first
for ax in axes:
    ax.axis("off")

# Plot
for i, gene in enumerate(G1_rest + G2_rest):
    sc.pl.umap(
        adata_from_raw,
        color=gene,
        cmap=umap_cmap,
        s=10,
        ax=axes[i],
        show=False,
        title=gene
    )
    axes[i].set_axis_off()
    axes[i].title.set_fontsize(20)
    axes[i].title.set_fontweight("bold")

plt.tight_layout()
plt.savefig('./Figure_plots/Figure_sup_G1G2DGE/umaps_rest.pdf')
plt.show()


In [ ]:
sc.pl.umap(adata, color = 'score_DGE_G2', cmap = 'Purples', size = 10, save= '_umapG2.pdf')

In [ ]:
sc.pl.umap(adata, color = 'score_DGE_G1', cmap = 'Purples', size = 10, save= '_umapG1.pdf')

In [ ]:
def plot_obs_scatter(adata, x_col, y_col, method="pearson", hue_col=None, palette="viridis", save_path=None, fontsize = 8):
    df = adata.obs.copy()
    corr = df[x_col].corr(df[y_col], method=method)
    plt.figure(figsize=(6, 6))
    ax = plt.gca()
    ax.grid(False)
    if hue_col and hue_col in df.columns:
        sc = sns.scatterplot(
            data=df, x=x_col, y=y_col,
            hue=hue_col, palette=palette,
            s=2, edgecolor=None, zorder=3, legend=False, 
            rasterized=True
        )
        sns.regplot(
            data=df, x=x_col, y=y_col,
            scatter=False,
            line_kws={"color": "black", "zorder": 4}
        )
        if pd.api.types.is_numeric_dtype(df[hue_col]):
            norm = plt.Normalize(vmin=df[hue_col].min(), vmax=df[hue_col].max())
            sm = plt.cm.ScalarMappable(cmap=palette, norm=norm)
            sm.set_array([])
            cbar = plt.colorbar(sm, ax=ax)
            cbar.set_label(hue_col, fontsize=fontsize)
            cbar.ax.tick_params(labelsize=fontsize)
    else:
        sns.regplot(
            data=df, x=x_col, y=y_col,
            scatter_kws={"s": 20, "alpha": 0.7, "zorder": 3},
            line_kws={"color": "black", "zorder": 4}
        )
    ax.tick_params(axis='both', labelsize=fontsize) 
    ax.set_xlabel(x_col, fontsize=fontsize)        
    ax.set_ylabel(y_col, fontsize=fontsize)         
    ax.text(
        0.05, 0.95, f"{method}: {corr:.3f}",
        transform=ax.transAxes, ha="left", va="top", 
        fontsize=fontsize + 2 
    )
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    
    plt.show()

In [ ]:
# define the colormap
colors = ['#f08080', '#4a3a4a', '#6a8faf', '#b0c4de', "#b0e0e6"]
stops  = [0, 25, 50, 255]
positions = [s / 255 for s in stops]

custom_cmap = mcolors.LinearSegmentedColormap.from_list(
    name="custom_cmap",
    colors=list(zip(positions, colors)),
    N=256)


In [ ]:
# check color map
gradient = np.linspace(0, 1, 256)
gradient = np.vstack((gradient, gradient))

plt.figure(figsize=(6, 1))
plt.title("Custom Purples Colormap")
plt.imshow(gradient, aspect='auto', cmap=custom_cmap)
plt.axis('off')
plt.show()

In [ ]:
plot_obs_scatter(adata, 'neoblast_score', 'score_DGE_G1', 
                 hue_col='n_counts', method="spearman", palette=custom_cmap, fontsize=16,
                save_path='./Figure_plots/'+ figure + "/correlation_neo_G1.pdf" )

In [ ]:
plot_obs_scatter(adata, 'neoblast_score', 'score_DGE_G2', 
                 hue_col='n_counts', method="spearman", palette=custom_cmap, fontsize=16,
                save_path= './Figure_plots/'+ figure + "/correlation_neo_G2.pdf" )

In [ ]:
plot_obs_scatter(adata, 'score_DGE_G1', 'score_DGE_G2', hue_col='n_counts', 
                 method="spearman", palette=custom_cmap, fontsize=16,
                    save_path= './Figure_plots/'+ figure + "/correlation_G1_G2.pdf" )

In [ ]:
# list of genes for the neoblast score
# 50 top markers of clusters 0 and 1 with wilcoxon
markers_w = pd.DataFrame(adata.uns['rank_genes_groups_wilcox_leiden_2.5']['names']).head(50)
li_neo_score = list(set(markers_w['0'].head(50).to_list() + markers_w['1'].head(50).to_list()))

In [ ]:
# list of genes for the G1 score
# genes differentially overexpressed in G1 and enriched in neoblasts
li_G1_score = list(adata.uns['2c_li_DGE'])

In [ ]:
# list of genes for the G2 score
# genes differentially overexpressed in G2 and enriched in neoblasts
li_G2_score = list(adata.uns['4c_li_DGE'])

In [ ]:
# overlap between G1 score and neoblast score
[i for i in li_G1_score if i in li_neo_score]

In [ ]:
# overlap between G2 score and neoblast score
[i for i in li_G2_score if i in li_neo_score]

In [ ]:
len(li_neo_score)

In [ ]:
len(li_G1_score)

In [ ]:
len(li_G2_score)